In [3]:
import requests

In [4]:
base_url = "http://localhost:5000"

coordinates = "-73.9855,40.7580;-73.9654,40.7829"

url = f"{base_url}/route/v1/driving/{coordinates}"

In [5]:
response = requests.get(
    url,
    params={"overview": "false"}
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'code': 'Ok', 'routes': [{'legs': [{'steps': [], 'weight': 504.4, 'summary': '', 'duration': 500.2, 'distance': 4788.6}], 'weight_name': 'routability', 'weight': 504.4, 'duration': 500.2, 'distance': 4788.6}], 'waypoints': [{'hint': 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA', 'location': [-73.985506, 40.758002], 'name': '7th Avenue', 'distance': 0.5530772553}, {'hint': 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA', 'location': [-73.965024, 40.783528], 'name': '', 'distance': 76.62317123}]}


In [6]:
data = response.json()

distance_km = data['routes'][0]['distance'] / 1000
duration_minutes = data['routes'][0]['duration'] / 60

print("Road distance:", round(distance_km, 2), "km")
print("Estimated duration:", round(duration_minutes, 2), "minutes")

Road distance: 4.79 km
Estimated duration: 8.34 minutes


In [7]:

import requests

BASE_URL = "http://localhost:5000"

def get_road_distance(
    pickup_lon, pickup_lat,
    dropoff_lon, dropoff_lat
):
    # 1. Construct the coordinates string
    coordinates = (
        f"{pickup_lon},{pickup_lat};"
        f"{dropoff_lon},{dropoff_lat}"
    )

    # 2. Construct the API URL
    url = f"{BASE_URL}/route/v1/driving/{coordinates}"

    # 3. Send the GET request
    response = requests.get(
        url,
        params={"overview": "false"},
        timeout=30
    )

    # 4. Convert response into JSON
    data = response.json()

    # 5. Check whether routing was successful
    if response.status_code != 200 or data.get("code") != "Ok":
        return None, None

    # 6. Extract distance and duration
    distance_km = data["routes"][0]["distance"] / 1000
    duration_minutes = data["routes"][0]["duration"] / 60

    # 7. Return distance and duration
    return distance_km, duration_minutes


Test the Function:

In [8]:

distance, duration = get_road_distance(
    -73.9855, 40.7580,
    -73.9654, 40.7829
)

print("Road distance:", round(distance, 2), "km")
print("Estimated duration:", round(duration, 2), "minutes")


Road distance: 4.79 km
Estimated duration: 8.34 minutes


In [9]:
import pandas as pd

# Load the processed dataset from Phase 2
df_processed = pd.read_parquet(
    "../data/processed/uber_haversine.parquet"
)

# Verify the dataset
print("Dataset shape:", df_processed.shape)
print("\nColumns:")
print(df_processed.columns.tolist())

df_processed.head()

Dataset shape: (195699, 11)

Columns:
['key', 'fare_amount', 'pickup_datetime', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'haversine_distance_km', 'is_zero_distance', 'has_nonpositive_fare']


,key,fare_amount,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,haversine_distance_km,is_zero_distance,has_nonpositive_fare
0,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06+00:00,1,-73.999817,40.738354,-73.999512,40.723217,1.683323,False,False
1,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56+00:00,1,-73.994355,40.728225,-73.994710,40.750325,2.457590,False,False
2,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00+00:00,1,-74.005043,40.740770,-73.962565,40.772647,5.036377,False,False
3,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21+00:00,3,-73.976124,40.790844,-73.965316,40.803349,1.661683,False,False
4,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00+00:00,5,-73.925023,40.744085,-73.973082,40.761247,4.475450,False,False


Check duplicate routes  
Your dataset contains 195,699 trips. However, multiple trips may have the same pickup and dropoff coordinates.  

For example, if 100 trips share the same pickup and dropoff locations, we only need to request that route from OSRM once.   We can reuse the result for all 100 trips.  
- Count unique coordinate pairs  

In [10]:
# Select pickup and dropoff coordinates
route_columns = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude"
]

# Count unique routes using exact coordinates
total_trips = len(df_processed)

unique_routes_exact = df_processed[route_columns].drop_duplicates()

total_unique_routes = len(unique_routes_exact)

print("Total trips:", total_trips)
print("Unique routes:", total_unique_routes)
print("Duplicate routes:", total_trips - total_unique_routes)

# Calculate potential API request reduction
reduction = (1 - total_unique_routes / total_trips) * 100

print("Potential API request reduction:", round(reduction, 2), "%")

Total trips: 195699
Unique routes: 195626
Duplicate routes: 73
Potential API request reduction: 0.04 %


But there is another possibility: two trips may have slightly different GPS coordinates while referring to practically the same pickup and dropoff locations.  

For example:  
- Trip A: -73.985500, 40.758000    
- Trip B: -73.985503, 40.758002  
These coordinates are extremely close.  
- Investigate coordinate rounding  
Let's check how many unique routes remain when we round coordinates to different decimal precisions.  

In [11]:
# Test different coordinate precisions
for decimals in [5, 4, 3]:

    rounded_routes = (
        df_processed[route_columns]
        .round(decimals)
        .drop_duplicates()
    )

    unique_count = len(rounded_routes)

    reduction = (
        1 - unique_count / total_trips
    ) * 100

    print(f"\nRounding to {decimals} decimal places:")
    print("Unique routes:", unique_count)
    print("Potential request reduction:", round(reduction, 2), "%")


Rounding to 5 decimal places:
Unique routes: 195623
Potential request reduction: 0.04 %

Rounding to 4 decimal places:
Unique routes: 195567
Potential request reduction: 0.07 %

Rounding to 3 decimal places:
Unique routes: 184643
Potential request reduction: 5.65 %


In [12]:
# Create a copy of the coordinate columns
route_lookup = df_processed[route_columns].copy()

# Round coordinates to 5 decimal places
route_lookup = route_lookup.round(5)

# Create unique route combinations
route_lookup = route_lookup.drop_duplicates().reset_index(drop=True)

# Assign a unique route ID
route_lookup["route_id"] = range(1, len(route_lookup) + 1)

print("Unique routes:", len(route_lookup))
print("\nSample routes:")
display(route_lookup.head())

Unique routes: 195623

Sample routes:


,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,route_id
0,-73.99982,40.73835,-73.99951,40.72322,1
1,-73.99436,40.72822,-73.99471,40.75032,2
2,-74.00504,40.74077,-73.96256,40.77265,3
3,-73.97612,40.79084,-73.96532,40.80335,4
4,-73.92502,40.74408,-73.97308,40.76125,5


# Map route IDs to taxi trips :  
The important thing here is that we will use rounded coordinates for matching, but preserve the original coordinates in df_processed for accurate OSRM routing.  

In [13]:
# Create rounded coordinate columns for matching
rounded_trip_coords = (
    df_processed[route_columns]
    .round(5)
    .add_suffix("_rounded")
)

# Prepare route lookup keys
route_lookup_keys = (
    route_lookup[route_columns]
    .add_suffix("_rounded")
)

route_lookup_keys["route_id"] = route_lookup["route_id"]

# Combine original trips with rounded matching coordinates
df_with_routes = pd.concat(
    [
        df_processed.reset_index(drop=True),
        rounded_trip_coords.reset_index(drop=True)
    ],
    axis=1
)

# Match every trip with its route ID
df_with_routes = df_with_routes.merge(
    route_lookup_keys,
    on=[
        "pickup_longitude_rounded",
        "pickup_latitude_rounded",
        "dropoff_longitude_rounded",
        "dropoff_latitude_rounded"
    ],
    how="left",
    validate="many_to_one"
)

# Remove temporary matching columns
df_with_routes.drop(
    columns=[
        "pickup_longitude_rounded",
        "pickup_latitude_rounded",
        "dropoff_longitude_rounded",
        "dropoff_latitude_rounded"
    ],
    inplace=True
)

C:\Users\USER\AppData\Local\Temp\ipykernel_14976\1687277202.py:39: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_with_routes.drop(


In [14]:
print("Original trips:", len(df_processed))
print("Trips with route IDs:", len(df_with_routes))

print(
    "Missing route IDs:",
    df_with_routes["route_id"].isna().sum()
)

print(
    "Unique route IDs:",
    df_with_routes["route_id"].nunique()
)

print("\nSample:")
display(
    df_with_routes[
        route_columns + ["route_id"]
    ].head()
)

Original trips: 195699
Trips with route IDs: 195699
Missing route IDs: 0
Unique route IDs: 195623

Sample:


,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,route_id
0,-73.999817,40.738354,-73.999512,40.723217,1
1,-73.994355,40.728225,-73.994710,40.750325,2
2,-74.005043,40.740770,-73.962565,40.772647,3
3,-73.976124,40.790844,-73.965316,40.803349,4
4,-73.925023,40.744085,-73.973082,40.761247,5


# Create the SQLite route cache  
Why SQLite?  
Imagine you process 50,000 routes today, but your computer shuts down.  

Without a cache, you might have to request those routes again. With SQLite, completed results remain saved, so you can resume later.  

In [15]:
import sqlite3

# Database location
DB_PATH = "../data/processed/osrm_route_cache.db"

# Connect to SQLite database
conn = sqlite3.connect(DB_PATH)

# Create route cache table
conn.execute("""
    CREATE TABLE IF NOT EXISTS routes (
        route_id INTEGER PRIMARY KEY,
        pickup_longitude REAL NOT NULL,
        pickup_latitude REAL NOT NULL,
        dropoff_longitude REAL NOT NULL,
        dropoff_latitude REAL NOT NULL,
        road_distance_km REAL,
        road_duration_min REAL,
        status TEXT NOT NULL DEFAULT 'pending'
    )
""")

conn.commit()

print("SQLite route cache created successfully!")

SQLite route cache created successfully!


In [16]:
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(routes)")

for column in cursor.fetchall():
    print(column[1], "-", column[2])

route_id - INTEGER
pickup_longitude - REAL
pickup_latitude - REAL
dropoff_longitude - REAL
dropoff_latitude - REAL
road_distance_km - REAL
road_duration_min - REAL
status - TEXT


In [17]:
# Prepare route records for insertion
route_records = route_lookup[
    [
        "route_id",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude"
    ]
].itertuples(index=False, name=None)

# Insert routes into SQLite
conn.executemany("""
    INSERT OR IGNORE INTO routes (
        route_id,
        pickup_longitude,
        pickup_latitude,
        dropoff_longitude,
        dropoff_latitude
    )
    VALUES (?, ?, ?, ?, ?)
""", route_records)

conn.commit()

print("Unique routes inserted successfully!")

Unique routes inserted successfully!


In [18]:
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM routes")
total_routes = cursor.fetchone()[0]

cursor.execute("""
    SELECT status, COUNT(*)
    FROM routes
    GROUP BY status
""")

print("Total routes in database:", total_routes)
print("\nRoute status:")
for status, count in cursor.fetchall():
    print(status, ":", count)

Total routes in database: 195623

Route status:
pending : 195613
success : 10


Build the batch-processing function  
We'll process routes in small batches rather than loading all routes into memory.

Create a routing progress monitor  
This function will show how many routes have been completed, failed, and remain pending.  

1. get_road_distance()  
Keep  

Sends one route request to OSRM and returns its result.  

2. process_route_batch()  
keep  

Processes a batch of routes and saves results into SQLite.  

3. run_routing_pipeline()  
Keep  

Calls process_route_batch() repeatedly and monitors progress.  

In [26]:
def show_routing_progress():

    cursor = conn.cursor()

    cursor.execute("""
        SELECT status, COUNT(*)
        FROM routes
        GROUP BY status
    """)

    results = dict(cursor.fetchall())

    total = sum(results.values())
    successful = results.get("success", 0)
    failed = results.get("failed", 0)
    pending = results.get("pending", 0)

    completed = successful + failed

    print("=" * 40)
    print("       OSRM ROUTING PROGRESS")
    print("=" * 40)

    print(f"Total routes      : {total:,}")
    print(f"Successful routes : {successful:,}")
    print(f"Failed routes     : {failed:,}")
    print(f"Pending routes    : {pending:,}")

    if total:
        progress = completed / total * 100
        print(f"Completion        : {progress:.2f}%")

    print("=" * 40)

In [27]:
show_routing_progress()

       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 20
Failed routes     : 0
Pending routes    : 195,603
Completion        : 0.01%


In [32]:
def process_route_batch(batch_size=100):

    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            route_id,
            pickup_longitude,
            pickup_latitude,
            dropoff_longitude,
            dropoff_latitude
        FROM routes
        WHERE status = 'pending'
        ORDER BY route_id
        LIMIT ?
    """, (batch_size,))

    routes = cursor.fetchall()

    if not routes:
        print("No pending routes remaining.")
        return

    successful = 0
    failed = 0
    retryable = 0

    for route in routes:

        route_id, pickup_lon, pickup_lat, dropoff_lon, dropoff_lat = route

        distance, duration, result = get_road_distance(
            pickup_lon, pickup_lat,
            dropoff_lon, dropoff_lat
        )

        if result == "success":

            cursor.execute("""
                UPDATE routes
                SET road_distance_km = ?,
                    road_duration_min = ?,
                    status = 'success'
                WHERE route_id = ?
            """, (distance, duration, route_id))

            successful += 1

        elif result == "unroutable":

            cursor.execute("""
                UPDATE routes
                SET status = 'failed'
                WHERE route_id = ?
            """, (route_id,))

            failed += 1

        else:
            # Leave the route pending for another attempt
            retryable += 1

    conn.commit()

    print("\nBatch completed!")
    print("Processed:", len(routes))
    print("Successful:", successful)
    print("Unroutable:", failed)
    print("Temporary errors:", retryable)

    show_routing_progress()

In [34]:
process_route_batch(batch_size=10)


Batch completed!
Processed: 10
Successful: 10
Unroutable: 0
Temporary errors: 0
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 540
Failed routes     : 0
Pending routes    : 195,083
Completion        : 0.28%


In [28]:
def run_routing_pipeline(
    batch_size=100,
    max_batches=5
):

    batch_number = 0

    print("Starting OSRM routing pipeline...\n")

    while batch_number < max_batches:

        cursor = conn.cursor()

        cursor.execute("""
            SELECT COUNT(*)
            FROM routes
            WHERE status = 'pending'
        """)

        pending = cursor.fetchone()[0]

        if pending == 0:
            print("All pending routes have been processed!")
            break

        print(f"\nProcessing batch {batch_number + 1}...")

        process_route_batch(batch_size=batch_size)

        batch_number += 1

        show_routing_progress()

    print("\nRouting pipeline execution finished.")

In [29]:
run_routing_pipeline(
    batch_size=100,
    max_batches=5
)

Starting OSRM routing pipeline...


Processing batch 1...

Batch completed!
Processed: 100
Successful: 100
Failed: 0
Total successful routes: 120
Pending routes: 195503
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 120
Failed routes     : 0
Pending routes    : 195,503
Completion        : 0.06%

Processing batch 2...

Batch completed!
Processed: 100
Successful: 100
Failed: 0
Total successful routes: 220
Pending routes: 195403
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 220
Failed routes     : 0
Pending routes    : 195,403
Completion        : 0.11%

Processing batch 3...

Batch completed!
Processed: 100
Successful: 100
Failed: 0
Total successful routes: 320
Pending routes: 195303
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 320
Failed routes     : 0
Pending routes    : 195,303
Completion        : 0.16%

Processing batch 4...

Batch completed!
Processed: 100
Successful: 100
Failed: 0
Total succ

In [36]:
## for test only

result = get_road_distance(
    -73.9855, 40.7580,
    -73.9654, 40.7829
)

print("Result:", result)

Result: (4.788600000000001, 8.336666666666666, 'success')


In [37]:
## For test only

process_route_batch(batch_size=10)


Batch completed!
Processed: 10
Successful: 10
Unroutable: 0
Temporary errors: 0
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 550
Failed routes     : 0
Pending routes    : 195,073
Completion        : 0.28%


In [38]:
##  For test only

show_routing_progress()

       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 550
Failed routes     : 0
Pending routes    : 195,073
Completion        : 0.28%


Benchmark routing performance  
Before processing nearly 195,000 routes, we should measure how quickly your local OSRM server handles requests.  

In [39]:
import time

test_batch_size = 50

start_time = time.time()

process_route_batch(batch_size=test_batch_size)

end_time = time.time()

elapsed_time = end_time - start_time

print("\nPerformance Report")
print("------------------")
print("Batch size:", test_batch_size)
print("Time taken:", round(elapsed_time, 2), "seconds")
print(
    "Average time per route:",
    round(elapsed_time / test_batch_size, 3),
    "seconds"
)


Batch completed!
Processed: 50
Successful: 50
Unroutable: 0
Temporary errors: 0
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 600
Failed routes     : 0
Pending routes    : 195,023
Completion        : 0.31%

Performance Report
------------------
Batch size: 50
Time taken: 0.36 seconds
Average time per route: 0.007 seconds


Why benchmark?  
We want to estimate:  
- Processing speed per route.  
- Approximate time for the entire dataset.  
- Whether increasing batch size would help.    
- Whether SQLite commits or OSRM requests are the bottleneck.  
Important: This test processes 50 additional routes, not the whole dataset.  

Check OSRM routing quality  
Before processing everything, let's verify whether OSRM is returning realistic road distances.  

In [40]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*),
        MIN(road_distance_km),
        MAX(road_distance_km),
        AVG(road_distance_km),
        MIN(road_duration_min),
        MAX(road_duration_min),
        AVG(road_duration_min)
    FROM routes
    WHERE status = 'success'
""")

stats = cursor.fetchone()

print("Successful routes:", stats[0])
print("Minimum road distance:", stats[1])
print("Maximum road distance:", stats[2])
print("Average road distance:", stats[3])
print("Minimum duration:", stats[4])
print("Maximum duration:", stats[5])
print("Average duration:", stats[6])

Successful routes: 600
Minimum road distance: 0.0
Maximum road distance: 29.2278
Average road distance: 4.333696166666667
Minimum duration: 0.0
Maximum duration: 31.470000000000002
Average duration: 7.494125


In [41]:
cursor.execute("""
    SELECT COUNT(*)
    FROM routes
    WHERE status = 'success'
      AND (
          road_distance_km <= 0
          OR road_duration_min <= 0
          OR road_distance_km > 100
          OR road_duration_min > 180
      )
""")

print("Suspicious routes:", cursor.fetchone()[0])

Suspicious routes: 6


Display the suspicious records

In [42]:
cursor.execute("""
    SELECT
        route_id,
        pickup_longitude,
        pickup_latitude,
        dropoff_longitude,
        dropoff_latitude,
        road_distance_km,
        road_duration_min
    FROM routes
    WHERE status = 'success'
      AND (
          road_distance_km <= 0
          OR road_duration_min <= 0
          OR road_distance_km > 100
          OR road_duration_min > 180
      )
""")

suspicious_routes = cursor.fetchall()

for route in suspicious_routes:
    print(route)

(6, -73.96902, 40.75591, -73.96902, 40.75591, 0.0, 0.0)
(47, -73.9935, 40.76469, -73.9935, 40.76469, 0.0, 0.0)
(82, -73.97506, 40.78882, -73.97506, 40.78882, 0.0, 0.0)
(161, -73.99294, 40.75778, -73.99294, 40.75778, 0.0, 0.0)
(352, -73.92871, 40.75348, -73.92871, 40.75348, 0.0, 0.0)
(455, -74.00272, 40.76064, -74.00272, 40.76064, 0.0, 0.0)


Check how many have zero distance versus zero duration

In [43]:
cursor.execute("""
    SELECT
        SUM(CASE WHEN road_distance_km = 0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN road_duration_min = 0 THEN 1 ELSE 0 END)
    FROM routes
    WHERE status = 'success'
""")

zero_stats = cursor.fetchone()

print("Zero-distance routes:", zero_stats[0])
print("Zero-duration routes:", zero_stats[1])

Zero-distance routes: 6
Zero-duration routes: 6


Validate the remaining route quality  
Let's check whether the road distance is generally greater than or equal to the Haversine distance.  

This is a useful quality check because road routes are normally longer than straight-line distances.  

In [44]:
# Merge successful OSRM results with original trip data
df_quality_check = df_with_routes.merge(
    pd.read_sql_query("""
        SELECT route_id, road_distance_km, road_duration_min
        FROM routes
        WHERE status = 'success'
    """, conn),
    on="route_id",
    how="inner"
)

# Calculate road-to-straight-line distance ratio
df_quality_check["distance_ratio"] = (
    df_quality_check["road_distance_km"] /
    df_quality_check["haversine_distance_km"].replace(0, float("nan"))
)

print("Successful trip records:", len(df_quality_check))

print("\nDistance ratio statistics:")
print(df_quality_check["distance_ratio"].describe())

print(
    "\nRoad distance shorter than Haversine:",
    (
        df_quality_check["road_distance_km"] <
        df_quality_check["haversine_distance_km"]
    ).sum()
)

Successful trip records: 600

Distance ratio statistics:
count    594.000000
mean       1.409650
std        0.928076
min        0.088419
25%        1.184202
50%        1.313261
75%        1.429178
max       15.882531
Name: distance_ratio, dtype: float64

Road distance shorter than Haversine: 11


In [45]:
shorter_routes = df_quality_check[
    df_quality_check["road_distance_km"] <
    df_quality_check["haversine_distance_km"]
].copy()

shorter_routes = shorter_routes[
    [
        "route_id",
        "haversine_distance_km",
        "road_distance_km",
        "road_duration_min",
        "distance_ratio"
    ]
].sort_values("distance_ratio")

display(shorter_routes)

,route_id,haversine_distance_km,road_distance_km,road_duration_min,distance_ratio
415,416,0.097264,0.0086,0.035000,0.088419
286,287,0.453429,0.4525,1.005000,0.997950
20,21,1.261388,1.2597,2.191667,0.998662
161,162,1.066523,1.0652,2.233333,0.998760
431,432,1.730938,1.7294,3.858333,0.999112
90,91,1.232284,1.2313,2.743333,0.999202
477,478,2.139018,2.1380,4.853333,0.999524
517,518,1.329695,1.3291,3.106667,0.999553
366,367,1.958453,1.9577,4.520000,0.999616
597,598,2.802430,2.8014,6.396667,0.999632


In [46]:
display(
    df_quality_check.nsmallest(
        5, "distance_ratio"
    )[
        [
            "route_id",
            "pickup_longitude",
            "pickup_latitude",
            "dropoff_longitude",
            "dropoff_latitude",
            "haversine_distance_km",
            "road_distance_km",
            "distance_ratio"
        ]
    ]
)

,route_id,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,haversine_distance_km,road_distance_km,distance_ratio
415,416,-73.787044,40.641077,-73.786471,40.641836,0.097264,0.0086,0.088419
286,287,-73.953837,40.775171,-73.956403,40.771586,0.453429,0.4525,0.997950
20,21,-74.010798,40.726085,-74.009767,40.737402,1.261388,1.2597,0.998662
161,162,-73.992667,40.753087,-73.986788,40.761582,1.066523,1.0652,0.998760
431,432,-73.994312,40.740935,-73.984237,40.754502,1.730938,1.7294,0.999112


Investigate route 416  
Let's inspect how OSRM snapped the pickup and dropoff coordinates.  

In [47]:
route_416 = df_quality_check[
    df_quality_check["route_id"] == 416
].iloc[0]

pickup_lon = route_416["pickup_longitude"]
pickup_lat = route_416["pickup_latitude"]

dropoff_lon = route_416["dropoff_longitude"]
dropoff_lat = route_416["dropoff_latitude"]

print("Pickup:", pickup_lon, pickup_lat)
print("Dropoff:", dropoff_lon, dropoff_lat)

Pickup: -73.787044 40.641077
Dropoff: -73.78647099999999 40.641836


In [48]:
coordinates = (
    f"{pickup_lon},{pickup_lat};"
    f"{dropoff_lon},{dropoff_lat}"
)

url = f"{BASE_URL}/route/v1/driving/{coordinates}"

response = requests.get(
    url,
    params={"overview": "false"},
    timeout=30
)

data = response.json()

print("OSRM status:", data["code"])

print("\nRoute distance:", data["routes"][0]["distance"], "metres")
print("Route duration:", data["routes"][0]["duration"], "seconds")

print("\nWaypoint information:")

for waypoint in data["waypoints"]:
    print("Original location:", waypoint["location"])
    print("Snapped road name:", waypoint["name"])
    print("Snapping distance:", waypoint["distance"], "metres")
    print("---")

OSRM status: Ok

Route distance: 8.8 metres
Route duration: 2.1 seconds

Waypoint information:
Original location: [-73.78725, 40.640735]
Snapped road name: 
Snapping distance: 41.78660202 metres
---
Original location: [-73.787155, 40.640702]
Snapped road name: 
Snapping distance: 138.5890288 metres
---


In [49]:
# Inspect routes with unusually high distance ratios

outlier_routes = df_quality_check.nlargest(
    10, "distance_ratio"
)[
    [
        "route_id",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "haversine_distance_km",
        "road_distance_km",
        "road_duration_min",
        "distance_ratio"
    ]
]

display(outlier_routes)

,route_id,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,haversine_distance_km,road_distance_km,road_duration_min,distance_ratio
566,567,-74.015265,40.708583,-74.013985,40.713280,0.533309,8.4703,11.020000,15.882531
444,445,-74.030417,40.807683,-74.027813,40.806813,0.239566,3.7141,4.556667,15.503444
312,313,-74.004350,40.751103,-74.004582,40.752137,0.116625,0.9660,1.980000,8.282988
372,373,-73.988257,40.737727,-73.986637,40.739952,0.282560,1.4496,3.755000,5.130246
97,98,-73.977633,40.753166,-73.981589,40.757359,0.573074,1.6192,3.076667,2.825466
469,470,-73.954566,40.719553,-73.949270,40.720892,0.470500,1.2890,3.116667,2.739637
499,500,-73.996127,40.758460,-73.999807,40.753539,0.628885,1.6767,3.171667,2.666149
383,384,-73.980082,40.765167,-73.982370,40.770162,0.587892,1.4348,3.633333,2.440584
354,355,-73.955365,40.764890,-73.957835,40.767198,0.330354,0.8023,2.173333,2.428607
497,498,-73.977043,40.795833,-73.989055,40.795602,1.011490,2.3420,4.028333,2.315397


# Run 3 batches with size of 1000

In [50]:
run_routing_pipeline(
    batch_size=1000,
    max_batches=3
)

Starting OSRM routing pipeline...


Processing batch 1...

Batch completed!
Processed: 1000
Successful: 1000
Unroutable: 0
Temporary errors: 0
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 1,600
Failed routes     : 0
Pending routes    : 194,023
Completion        : 0.82%
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 1,600
Failed routes     : 0
Pending routes    : 194,023
Completion        : 0.82%

Processing batch 2...

Batch completed!
Processed: 1000
Successful: 1000
Unroutable: 0
Temporary errors: 0
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 2,600
Failed routes     : 0
Pending routes    : 193,023
Completion        : 1.33%
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 2,600
Failed routes     : 0
Pending routes    : 193,023
Completion        : 1.33%

Processing batch 3...

Batch completed!
Processed: 1000
Successful: 1000
Unroutable: 0
Temporary errors: 0
     

In [51]:
show_routing_progress()

       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 3,600
Failed routes     : 0
Pending routes    : 192,023
Completion        : 1.84%


# Now for big batches (193 batches with 1000 batch size)

In [52]:
run_routing_pipeline(
    batch_size=1000,
    max_batches=193
)

Starting OSRM routing pipeline...


Processing batch 1...

Batch completed!
Processed: 1000
Successful: 1000
Unroutable: 0
Temporary errors: 0
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 4,600
Failed routes     : 0
Pending routes    : 191,023
Completion        : 2.35%
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 4,600
Failed routes     : 0
Pending routes    : 191,023
Completion        : 2.35%

Processing batch 2...

Batch completed!
Processed: 1000
Successful: 1000
Unroutable: 0
Temporary errors: 0
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 5,600
Failed routes     : 0
Pending routes    : 190,023
Completion        : 2.86%
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 5,600
Failed routes     : 0
Pending routes    : 190,023
Completion        : 2.86%

Processing batch 3...

Batch completed!
Processed: 1000
Successful: 1000
Unroutable: 0
Temporary errors: 0
     

In [53]:
show_routing_progress()

       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 195,622
Failed routes     : 0
Pending routes    : 1
Completion        : 100.00%


In [55]:
cursor.execute("""
    SELECT *
    FROM routes
    WHERE status = 'pending'
""")

pending_routes = cursor.fetchall()

for route in pending_routes:
    print(route)

(45162, -74.02597, 40.73832, -74.01399, 40.72692, None, None, 'pending')


In [56]:
process_route_batch(batch_size=1)

Attempt 1 failed: 400 Client Error: Bad Request for url: http://localhost:5000/route/v1/driving/-74.02597,40.73832;-74.01399,40.72692?overview=false
Attempt 2 failed: 400 Client Error: Bad Request for url: http://localhost:5000/route/v1/driving/-74.02597,40.73832;-74.01399,40.72692?overview=false
Attempt 3 failed: 400 Client Error: Bad Request for url: http://localhost:5000/route/v1/driving/-74.02597,40.73832;-74.01399,40.72692?overview=false

Batch completed!
Processed: 1
Successful: 0
Unroutable: 0
Temporary errors: 1
       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 195,622
Failed routes     : 0
Pending routes    : 1
Completion        : 100.00%


In [57]:
show_routing_progress()

       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 195,622
Failed routes     : 0
Pending routes    : 1
Completion        : 100.00%


In [58]:
route_id = 45162

cursor.execute("""
    SELECT
        pickup_longitude,
        pickup_latitude,
        dropoff_longitude,
        dropoff_latitude
    FROM routes
    WHERE route_id = ?
""", (route_id,))

pickup_lon, pickup_lat, dropoff_lon, dropoff_lat = cursor.fetchone()

coordinates = (
    f"{pickup_lon},{pickup_lat};"
    f"{dropoff_lon},{dropoff_lat}"
)

url = f"{BASE_URL}/route/v1/driving/{coordinates}"

response = requests.get(
    url,
    params={"overview": "false"},
    timeout=30
)

print("HTTP status:", response.status_code)
print("OSRM response:")
print(response.json())

HTTP status: 400
OSRM response:
{'message': 'No route found between points', 'code': 'NoRoute'}


In [59]:
cursor.execute("""
    UPDATE routes
    SET status = 'failed'
    WHERE route_id = 45162
""")

conn.commit()

print("Route 45162 classified as failed.")

Route 45162 classified as failed.


In [60]:
show_routing_progress()

       OSRM ROUTING PROGRESS
Total routes      : 195,623
Successful routes : 195,622
Failed routes     : 1
Pending routes    : 0
Completion        : 100.00%


Merge and export the final dataset  
Now we'll combine the OSRM results with your original taxi records.  

In [61]:
# Load successful OSRM route results
df_routes = pd.read_sql_query("""
    SELECT
        route_id,
        road_distance_km,
        road_duration_min,
        status
    FROM routes
""", conn)

print("Route lookup shape:", df_routes.shape)
print("\nRoute status:")
print(df_routes["status"].value_counts())

display(df_routes.head())

Route lookup shape: (195623, 4)

Route status:
status
success    195622
failed          1
Name: count, dtype: int64


,route_id,road_distance_km,road_duration_min,status
0,1,2.0969,4.763333,success
1,2,3.4000,7.663333,success
2,3,6.4174,12.993333,success
3,4,2.2875,3.820000,success
4,5,4.9971,7.368333,success


# Merge OSRM results with taxi data

In [62]:
# Merge OSRM results with the original taxi trips
df_final = df_with_routes.merge(
    df_routes,
    on="route_id",
    how="left",
    validate="many_to_one"
)

print("Merge completed successfully!")
print("Final dataset shape:", df_final.shape)

display(df_final.head())

Merge completed successfully!
Final dataset shape: (195699, 15)


,key,fare_amount,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,haversine_distance_km,is_zero_distance,has_nonpositive_fare,route_id,road_distance_km,road_duration_min,status
0,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06+00:00,1,-73.999817,40.738354,-73.999512,40.723217,1.683323,False,False,1,2.0969,4.763333,success
1,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56+00:00,1,-73.994355,40.728225,-73.994710,40.750325,2.457590,False,False,2,3.4000,7.663333,success
2,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00+00:00,1,-74.005043,40.740770,-73.962565,40.772647,5.036377,False,False,3,6.4174,12.993333,success
3,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21+00:00,3,-73.976124,40.790844,-73.965316,40.803349,1.661683,False,False,4,2.2875,3.820000,success
4,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00+00:00,5,-73.925023,40.744085,-73.973082,40.761247,4.475450,False,False,5,4.9971,7.368333,success


Why left merge?  
We want to preserve every original taxi trip, even if its route failed.  

The one failed route will have missing road-distance and duration values. We should not silently delete that trip.

In [63]:
print("Original taxi records:", len(df_processed))
print("Final taxi records:", len(df_final))

print("\nMissing values:")
print(
    df_final[
        ["road_distance_km", "road_duration_min"]
    ].isna().sum()
)

print("\nRoute status:")
print(df_final["status"].value_counts(dropna=False))

print("\nDuplicate route IDs:")
print(df_final["route_id"].duplicated().sum())

Original taxi records: 195699
Final taxi records: 195699

Missing values:
road_distance_km     1
road_duration_min    1
dtype: int64

Route status:
status
success    195698
failed          1
Name: count, dtype: int64

Duplicate route IDs:
76


# Final dataset quality validation  
Before exporting, let's verify that our newly generated features contain valid values.  

In [64]:
# 1. Check negative road distances
negative_distances = (
    df_final["road_distance_km"] < 0
).sum()

# 2. Check negative durations
negative_durations = (
    df_final["road_duration_min"] < 0
).sum()

# 3. Check successful routes with missing values
missing_success = df_final[
    (df_final["status"] == "success") &
    (
        df_final["road_distance_km"].isna() |
        df_final["road_duration_min"].isna()
    )
].shape[0]

# 4. Check failed routes
failed_routes = (
    df_final["status"] == "failed"
).sum()

# 5. Check missing route IDs
missing_route_ids = df_final["route_id"].isna().sum()

# Display results
print("Negative road distances:", negative_distances)
print("Negative durations:", negative_durations)
print("Successful routes with missing values:", missing_success)
print("Failed routes:", failed_routes)
print("Missing route IDs:", missing_route_ids)

print("\nRoad distance statistics:")
print(df_final["road_distance_km"].describe())

print("\nRoad duration statistics:")
print(df_final["road_duration_min"].describe())

Negative road distances: 0
Negative durations: 0
Successful routes with missing values: 0
Failed routes: 1
Missing route IDs: 0

Road distance statistics:
count    195698.000000
mean          4.460250
std           4.950637
min           0.000000
25%           1.714800
50%           2.845800
75%           5.062900
max          57.134200
Name: road_distance_km, dtype: float64

Road duration statistics:
count    195698.000000
mean          7.638305
std           5.991343
min           0.000000
25%           3.648333
50%           5.878333
75%           9.735000
max          60.978333
Name: road_duration_min, dtype: float64


Export the final Parquet dataset

In [65]:
# Define output path
output_path = "../data/processed/uber_with_road_distance.parquet"

# Save final dataset
df_final.to_parquet(
    output_path,
    index=False,
    engine="pyarrow"
)

print("Final dataset exported successfully!")
print("File:", output_path)
print("Rows:", len(df_final))
print("Columns:", len(df_final.columns))

Final dataset exported successfully!
File: ../data/processed/uber_with_road_distance.parquet
Rows: 195699
Columns: 15


In [66]:
# Reload exported Parquet file
df_exported = pd.read_parquet(output_path)

print("Reloaded shape:", df_exported.shape)
print("\nColumns:")
print(df_exported.columns.tolist())

print("\nMissing values:")
print(
    df_exported[
        ["road_distance_km", "road_duration_min"]
    ].isna().sum()
)

print("\nFirst five records:")
display(df_exported.head())

Reloaded shape: (195699, 15)

Columns:
['key', 'fare_amount', 'pickup_datetime', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'haversine_distance_km', 'is_zero_distance', 'has_nonpositive_fare', 'route_id', 'road_distance_km', 'road_duration_min', 'status']

Missing values:
road_distance_km     1
road_duration_min    1
dtype: int64

First five records:


,key,fare_amount,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,haversine_distance_km,is_zero_distance,has_nonpositive_fare,route_id,road_distance_km,road_duration_min,status
0,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06+00:00,1,-73.999817,40.738354,-73.999512,40.723217,1.683323,False,False,1,2.0969,4.763333,success
1,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56+00:00,1,-73.994355,40.728225,-73.994710,40.750325,2.457590,False,False,2,3.4000,7.663333,success
2,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00+00:00,1,-74.005043,40.740770,-73.962565,40.772647,5.036377,False,False,3,6.4174,12.993333,success
3,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21+00:00,3,-73.976124,40.790844,-73.965316,40.803349,1.661683,False,False,4,2.2875,3.820000,success
4,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00+00:00,5,-73.925023,40.744085,-73.973082,40.761247,4.475450,False,False,5,4.9971,7.368333,success


Work on DateTime

In [67]:
pd.to_datetime(
    df_processed["pickup_datetime"],
    utc=True
)

0        2015-05-07 19:52:06+00:00
1        2009-07-17 20:04:56+00:00
2        2009-08-24 21:45:00+00:00
3        2009-06-26 08:22:21+00:00
4        2014-08-28 17:47:00+00:00
                    ...           
195694   2012-10-28 10:49:00+00:00
195695   2014-03-14 01:09:00+00:00
195696   2009-06-29 00:42:00+00:00
195697   2015-05-20 14:56:25+00:00
195698   2010-05-15 04:08:00+00:00
Name: pickup_datetime, Length: 195699, dtype: datetime64[us, UTC]

In [68]:
df_raw_datetime = pd.read_csv(
    "../data/raw/uber.csv",
    usecols=["pickup_datetime"],
    nrows=10
)

print("Original data type:")
print(df_raw_datetime.dtypes)

print("\nOriginal datetime samples:")
display(df_raw_datetime)

Original data type:
pickup_datetime    str
dtype: object

Original datetime samples:


,pickup_datetime
0,2015-05-07 19:52:06 UTC
1,2009-07-17 20:04:56 UTC
2,2009-08-24 21:45:00 UTC
3,2009-06-26 08:22:21 UTC
4,2014-08-28 17:47:00 UTC
5,2011-02-12 02:27:09 UTC
6,2014-10-12 07:04:00 UTC
7,2012-12-11 13:52:00 UTC
8,2012-02-17 09:32:00 UTC
9,2012-03-29 19:06:00 UTC


# Convert UTC timestamps to New York local time

In [69]:
# Convert UTC timestamps to New York local time
df_exported["pickup_datetime_ny"] = (
    df_exported["pickup_datetime"]
    .dt.tz_convert("America/New_York")
)

print("Original timezone:",
      df_exported["pickup_datetime"].dt.tz)

print("New timezone:",
      df_exported["pickup_datetime_ny"].dt.tz)

display(
    df_exported[
        ["pickup_datetime", "pickup_datetime_ny"]
    ].head(10)
)

Original timezone: UTC
New timezone: America/New_York


,pickup_datetime,pickup_datetime_ny
0,2015-05-07 19:52:06+00:00,2015-05-07 15:52:06-04:00
1,2009-07-17 20:04:56+00:00,2009-07-17 16:04:56-04:00
2,2009-08-24 21:45:00+00:00,2009-08-24 17:45:00-04:00
3,2009-06-26 08:22:21+00:00,2009-06-26 04:22:21-04:00
4,2014-08-28 17:47:00+00:00,2014-08-28 13:47:00-04:00
5,2011-02-12 02:27:09+00:00,2011-02-11 21:27:09-05:00
6,2014-10-12 07:04:00+00:00,2014-10-12 03:04:00-04:00
7,2012-02-17 09:32:00+00:00,2012-02-17 04:32:00-05:00
8,2012-03-29 19:06:00+00:00,2012-03-29 15:06:00-04:00
9,2015-05-22 17:32:27+00:00,2015-05-22 13:32:27-04:00


In [70]:
print("Missing New York timestamps:",
      df_exported["pickup_datetime_ny"].isna().sum())

print("\nNew York datetime dtype:")
print(df_exported["pickup_datetime_ny"].dtype)

Missing New York timestamps: 0

New York datetime dtype:
datetime64[us, America/New_York]


Extract time-based features  
These features help a machine learning model understand how taxi fares vary depending on pickup time.  

In [71]:
# Extract time-based features

df_exported["pickup_hour"] = (
    df_exported["pickup_datetime_ny"].dt.hour
)

df_exported["pickup_day"] = (
    df_exported["pickup_datetime_ny"].dt.day
)

df_exported["pickup_day_of_week"] = (
    df_exported["pickup_datetime_ny"].dt.dayofweek
)

df_exported["pickup_month"] = (
    df_exported["pickup_datetime_ny"].dt.month
)

df_exported["pickup_year"] = (
    df_exported["pickup_datetime_ny"].dt.year
)

# Weekend indicator: Saturday = 5, Sunday = 6
df_exported["is_weekend"] = (
    df_exported["pickup_day_of_week"].isin([5, 6])
).astype(int)

# Rush-hour indicator: weekday morning and evening
df_exported["is_rush_hour"] = (
    (df_exported["pickup_day_of_week"] < 5) &
    (
        df_exported["pickup_hour"].between(7, 9) |
        df_exported["pickup_hour"].between(16, 19)
    )
).astype(int)

print("Time-based features created successfully!")

display(
    df_exported[
        [
            "pickup_datetime_ny",
            "pickup_hour",
            "pickup_day",
            "pickup_day_of_week",
            "pickup_month",
            "pickup_year",
            "is_weekend",
            "is_rush_hour"
        ]
    ].head(10)
)

Time-based features created successfully!


C:\Users\USER\AppData\Local\Temp\ipykernel_14976\3205020544.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_exported["pickup_datetime_ny"].dt.dayofweek


,pickup_datetime_ny,pickup_hour,pickup_day,pickup_day_of_week,pickup_month,pickup_year,is_weekend,is_rush_hour
0,2015-05-07 15:52:06-04:00,15,7,3,5,2015,0,0
1,2009-07-17 16:04:56-04:00,16,17,4,7,2009,0,1
2,2009-08-24 17:45:00-04:00,17,24,0,8,2009,0,1
3,2009-06-26 04:22:21-04:00,4,26,4,6,2009,0,0
4,2014-08-28 13:47:00-04:00,13,28,3,8,2014,0,0
5,2011-02-11 21:27:09-05:00,21,11,4,2,2011,0,0
6,2014-10-12 03:04:00-04:00,3,12,6,10,2014,1,0
7,2012-02-17 04:32:00-05:00,4,17,4,2,2012,0,0
8,2012-03-29 15:06:00-04:00,15,29,3,3,2012,0,0
9,2015-05-22 13:32:27-04:00,13,22,4,5,2015,0,0


In [72]:
print("Weekend distribution:")
print(df_exported["is_weekend"].value_counts())

print("\nRush-hour distribution:")
print(df_exported["is_rush_hour"].value_counts())

print("\nPickup hour distribution:")
print(df_exported["pickup_hour"].value_counts().sort_index())

Weekend distribution:
is_weekend
0    144082
1     51617
Name: count, dtype: int64

Rush-hour distribution:
is_rush_hour
0    144487
1     51212
Name: count, dtype: int64

Pickup hour distribution:
pickup_hour
0      2176
1      2620
2      5112
3      7625
4      9079
5      9047
6      8943
7      9392
8      9827
9      9677
10     9496
11     8965
12     8537
13    10394
14    11911
15    12002
16    11440
17    11002
18    10383
19     8859
20     7044
21     5351
22     3965
23     2852
Name: count, dtype: int64


Create geographical and distance-based features  
Now we'll create features that describe the trip itself.  

These are particularly important for fare prediction.  

In [73]:
# Calculate road distance ratio
df_exported["road_distance_ratio"] = (
    df_exported["road_distance_km"] /
    df_exported["haversine_distance_km"].replace(0, float("nan"))
)

# Calculate difference between road and straight-line distance
df_exported["distance_difference_km"] = (
    df_exported["road_distance_km"] -
    df_exported["haversine_distance_km"]
)

print("Distance-based features created!")

Distance-based features created!


In [74]:
# Estimated average speed in km/h
df_exported["estimated_speed_kmh"] = (
    df_exported["road_distance_km"] /
    (df_exported["road_duration_min"] / 60)
)

# Replace infinite values with NaN
df_exported["estimated_speed_kmh"] = (
    df_exported["estimated_speed_kmh"].replace(
        [float("inf"), -float("inf")], float("nan")
    )
)

print("Estimated speed feature created!")

Estimated speed feature created!


In [75]:
display(
    df_exported[
        [
            "haversine_distance_km",
            "road_distance_km",
            "road_duration_min",
            "road_distance_ratio",
            "distance_difference_km",
            "estimated_speed_kmh"
        ]
    ].head(10)
)

,haversine_distance_km,road_distance_km,road_duration_min,road_distance_ratio,distance_difference_km,estimated_speed_kmh
0,1.683323,2.0969,4.763333,1.245691,0.413577,26.413016
1,2.457590,3.4000,7.663333,1.383469,0.942410,26.620270
2,5.036377,6.4174,12.993333,1.274210,1.381023,29.633966
3,1.661683,2.2875,3.820000,1.376616,0.625817,35.929319
4,4.475450,4.9971,7.368333,1.116558,0.521650,40.691156
5,0.000000,0.0000,0.000000,NaN,0.000000,NaN
6,11.731015,14.7555,17.660000,1.257820,3.024485,50.131937
7,2.332711,3.2279,6.826667,1.383755,0.895189,28.370215
8,4.889417,6.1205,12.111667,1.251785,1.231083,30.320352
9,2.250858,2.3824,5.315000,1.058441,0.131542,26.894450


In [76]:
print(
    df_exported[
        [
            "road_distance_ratio",
            "distance_difference_km",
            "estimated_speed_kmh"
        ]
    ].describe()
)

       road_distance_ratio  distance_difference_km  estimated_speed_kmh
count        193724.000000           195698.000000        193425.000000
mean              1.722688                1.143254            31.399684
std              37.924921                1.612566             7.126912
min               0.000000              -23.030299             0.000000
25%               1.188872                0.340340            26.348071
50%               1.325755                0.656450            29.299446
75%               1.450729                1.209025            34.657440
max           11052.903264               36.795923           100.800000


# Create geographical features  
Now let's create four basic features from pickup and dropoff coordinates:  
1. pickup_latitude, pickup_longitude  
2. dropoff_latitude, dropoff_longitude  
We already have these coordinates, so we don't need to create anything new from them yet.  

Instead, let's create pickup and dropoff location clusters using geographic coordinates. These can help the model learn common taxi pickup/dropoff areas.  

For now, we'll use a simple approach: round coordinates to three decimal places.  

In [77]:
df_exported["pickup_location_lat"] = (
    df_exported["pickup_latitude"].round(3)
)

df_exported["pickup_location_lon"] = (
    df_exported["pickup_longitude"].round(3)
)

df_exported["dropoff_location_lat"] = (
    df_exported["dropoff_latitude"].round(3)
)

df_exported["dropoff_location_lon"] = (
    df_exported["dropoff_longitude"].round(3)
)

In [78]:
print("Unique pickup locations:",
      df_exported[
          ["pickup_location_lat", "pickup_location_lon"]
      ].drop_duplicates().shape[0])

print("Unique dropoff locations:",
      df_exported[
          ["dropoff_location_lat", "dropoff_location_lon"]
      ].drop_duplicates().shape[0])

Unique pickup locations: 7572
Unique dropoff locations: 13669


# Feature engineering validation  
Before moving to model preparation, let's check whether our newly created features contain unexpected missing or infinite values.  

In [79]:
import numpy as np

# Check infinite values
numeric_df = df_exported.select_dtypes(include="number")

print("Infinite values:")
print(np.isinf(numeric_df).sum().sum())

# Check missing values in engineered features
engineered_features = [
    "pickup_hour",
    "pickup_day",
    "pickup_day_of_week",
    "pickup_month",
    "pickup_year",
    "is_weekend",
    "is_rush_hour",
    "road_distance_ratio",
    "distance_difference_km",
    "estimated_speed_kmh",
    "pickup_location_lat",
    "pickup_location_lon",
    "dropoff_location_lat",
    "dropoff_location_lon"
]

print("\nMissing values:")
print(df_exported[engineered_features].isnull().sum())

Infinite values:
0

Missing values:
pickup_hour                  0
pickup_day                   0
pickup_day_of_week           0
pickup_month                 0
pickup_year                  0
is_weekend                   0
is_rush_hour                 0
road_distance_ratio       1975
distance_difference_km       1
estimated_speed_kmh       2274
pickup_location_lat          0
pickup_location_lon          0
dropoff_location_lat         0
dropoff_location_lon         0
dtype: int64


1. Understanding missing values  
Feature	Missing	Reason  
road_distance_ratio	1,975	Zero Haversine distance or unavailable road distance  
distance_difference_km	1	One failed OSRM route  
estimated_speed_kmh	2,274	Zero duration or unavailable road distance/duration  

These are expected edge cases, not necessarily data corruption.  
2. What should we do?  
We should not delete these 2,274 records just because some engineered features are missing.  
 
Instead:  
- Keep the original trip records.  
- Preserve missing values for now.  
- Later, handle them during model preparation.   
- Consider adding indicators for missing or zero-distance values.  
This prevents unnecessary loss of training data.  

In [80]:
output_path = "../data/processed/uber_feature_engineered.parquet"

df_exported.to_parquet(
    output_path,
    index=False
)

print("Dataset saved successfully!")
print("Shape:", df_exported.shape)
print("File:", output_path)

Dataset saved successfully!
Shape: (195699, 30)
File: ../data/processed/uber_feature_engineered.parquet


In [81]:
df_check = pd.read_parquet(output_path)

print("Reloaded shape:", df_check.shape)
print("Total columns:", len(df_check.columns))

Reloaded shape: (195699, 30)
Total columns: 30


Prepare the dataset for Machine Learning
Before training a model, we need to:
1. Review all 30 columns.
2. Identify target, predictor and technical columns.
3. Handle missing values and suspicious records.
4. Select useful features and avoid data leakage.
5. Split the dataset into training and testing sets.

In [82]:
print("Dataset shape:", df_exported.shape)

for i, col in enumerate(df_exported.columns, start=1):
    print(f"{i}. {col} - {df_exported[col].dtype}")

Dataset shape: (195699, 30)
1. key - str
2. fare_amount - float64
3. pickup_datetime - datetime64[us, UTC]
4. passenger_count - int64
5. pickup_longitude - float64
6. pickup_latitude - float64
7. dropoff_longitude - float64
8. dropoff_latitude - float64
9. haversine_distance_km - float64
10. is_zero_distance - bool
11. has_nonpositive_fare - bool
12. route_id - int64
13. road_distance_km - float64
14. road_duration_min - float64
15. status - str
16. pickup_datetime_ny - datetime64[us, America/New_York]
17. pickup_hour - int32
18. pickup_day - int32
19. pickup_day_of_week - int32
20. pickup_month - int32
21. pickup_year - int32
22. is_weekend - int64
23. is_rush_hour - int64
24. road_distance_ratio - float64
25. distance_difference_km - float64
26. estimated_speed_kmh - float64
27. pickup_location_lat - float64
28. pickup_location_lon - float64
29. dropoff_location_lat - float64
30. dropoff_location_lon - float64


One modelling decision  
We should also be careful about estimated_speed_kmh: it is calculated from OSRM's estimated road distance and duration. It is not the actual historical speed of the taxi.  

We'll retain it as a candidate feature and evaluate its usefulness later.  
Check target variable quality  
Before creating the ML dataset, let's understand the fare distribution.

In [83]:
print("Fare statistics:")
print(df_exported["fare_amount"].describe())

print("\nNon-positive fares:")
print((df_exported["fare_amount"] <= 0).sum())

print("\nVery high fares (above $100):")
print((df_exported["fare_amount"] > 100).sum())

print("\nFare distribution:")
display(
    df_exported["fare_amount"]
    .value_counts()
    .sort_index()
    .head(20)
)

Fare statistics:
count    195699.000000
mean         11.338099
std           9.778040
min         -52.000000
25%           6.000000
50%           8.500000
75%          12.500000
max         499.000000
Name: fare_amount, dtype: float64

Non-positive fares:
19

Very high fares (above $100):
75

Fare distribution:


fare_amount
-52.00      1
-50.50      1
-49.57      1
-23.70      1
-10.90      1
-7.30       1
-6.90       1
-5.70       1
-5.50       1
-5.00       1
-3.50       3
-3.30       1
-3.00       2
 0.00       3
 0.01       2
 0.11       1
 2.50     715
 2.90     469
 3.00     347
 3.25       1
Name: count, dtype: int64

In [84]:
print("Total records:", len(df_exported))

print("Negative fares:",
      (df_exported["fare_amount"] < 0).sum())

print("Zero fares:",
      (df_exported["fare_amount"] == 0).sum())

print("Non-positive fares:",
      (df_exported["fare_amount"] <= 0).sum())

print("Fares above $100:",
      (df_exported["fare_amount"] > 100).sum())

print("Fares above $200:",
      (df_exported["fare_amount"] > 200).sum())

print("\nFare statistics:")
print(df_exported["fare_amount"].describe())

Total records: 195699
Negative fares: 16
Zero fares: 3
Non-positive fares: 19
Fares above $100: 75
Fares above $200: 4

Fare statistics:
count    195699.000000
mean         11.338099
std           9.778040
min         -52.000000
25%           6.000000
50%           8.500000
75%          12.500000
max         499.000000
Name: fare_amount, dtype: float64


For our first ML dataset:  
- Remove 19 records with fare <= 0.  
- Keep positive fares above $100 for now.  
- Don't automatically remove the $499 fare.  
- Later, evaluate extreme positive fares using distance and trip characteristics.  
Why remove non-positive fares? Our objective is to predict the amount a passenger should pay for a normal taxi trip. Negative and zero fares are not suitable target values for that objective.  

This removes only 0.0097% of the dataset.  

Create a clean modelling DataFrame  
Let's create a separate DataFrame so the original dataset remains unchanged.  

In [85]:
df_model = df_exported[
    df_exported["fare_amount"] > 0
].copy()

print("Original records:", len(df_exported))
print("Clean records:", len(df_model))
print("Removed records:", len(df_exported) - len(df_model))

print("\nRemaining non-positive fares:",
      (df_model["fare_amount"] <= 0).sum())

print("\nNew dataset shape:", df_model.shape)

Original records: 195699
Clean records: 195680
Removed records: 19

Remaining non-positive fares: 0

New dataset shape: (195680, 30)


In [86]:
missing_report = (
    df_model.isnull()
    .sum()
    .to_frame(name="missing_count")
)

missing_report["missing_percentage"] = (
    missing_report["missing_count"] / len(df_model) * 100
)

missing_report = missing_report[
    missing_report["missing_count"] > 0
].sort_values("missing_count", ascending=False)

display(missing_report)

,missing_count,missing_percentage
estimated_speed_kmh,2273,1.161590
road_distance_ratio,1974,1.008790
road_distance_km,1,0.000511
road_duration_min,1,0.000511
distance_difference_km,1,0.000511


Our missing-value strategy 
I recommend: 

| Feature | Strategy | Reason |
| --- | --- | --- |
| road_distance_km | Median imputation | Only one failed route |
| road_duration_min | Median imputation | Only one failed route |
| distance_difference_km | Median imputation | Only one missing value |
| road_distance_ratio | Keep NaN initially | Ratios can be misleading for zero-distance trips |
| estimated_speed_kmh | Keep NaN initially | Zero-duration trips have undefined speed |   

In [87]:
failed_trip = df_model[
    df_model["road_distance_km"].isnull()
]

display(
    failed_trip[
        [
            "fare_amount",
            "pickup_longitude",
            "pickup_latitude",
            "dropoff_longitude",
            "dropoff_latitude",
            "haversine_distance_km",
            "road_distance_km",
            "road_duration_min",
            "distance_difference_km"
        ]
    ]
)

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,haversine_distance_km,road_distance_km,road_duration_min,distance_difference_km
45172,8.9,-74.025967,40.738315,-74.013992,40.726922,1.619564,NaN,NaN,NaN


Our decision  
Keep this trip  
I recommend keeping the record and preserving its missing routing values.  

We should not immediately replace them with median values. We'll handle imputation after the train/test split, using statistics learned from the training set only. This avoids data leakage.  

In [88]:
df_model["osrm_route_missing"] = (
    df_model["road_distance_km"].isnull()
).astype(int)

print(
    df_model["osrm_route_missing"].value_counts()
)

osrm_route_missing
0    195679
1         1
Name: count, dtype: int64


Create missing-value indicators

In [89]:
df_model["road_distance_ratio_missing"] = (
    df_model["road_distance_ratio"].isnull()
).astype(int)

df_model["estimated_speed_missing"] = (
    df_model["estimated_speed_kmh"].isnull()
).astype(int)

print("Missing-value indicators:")
print(
    df_model[
        [
            "osrm_route_missing",
            "road_distance_ratio_missing",
            "estimated_speed_missing"
        ]
    ].sum()
)

Missing-value indicators:
osrm_route_missing                1
road_distance_ratio_missing    1974
estimated_speed_missing        2273
dtype: int64
